# Phần 1 — MCQ Datathon 2026
Tính trực tiếp từ dữ liệu cho 10 câu trắc nghiệm.

**Cấu trúc notebook:** mỗi câu một cell riêng để dễ chạy lại / sửa / kiểm tra.

## 0 — Setup

In [1]:
import pandas as pd
import numpy as np
from pathlib import Path

DATA = Path('dataset')  # đồng bộ với cấu trúc repo hiện tại

orders      = pd.read_csv(DATA/'orders.csv', parse_dates=['order_date'])
order_items = pd.read_csv(DATA/'order_items.csv', low_memory=False)
products    = pd.read_csv(DATA/'products.csv')
customers   = pd.read_csv(DATA/'customers.csv')
returns     = pd.read_csv(DATA/'returns.csv')
reviews     = pd.read_csv(DATA/'reviews.csv')
payments    = pd.read_csv(DATA/'payments.csv')
shipments   = pd.read_csv(DATA/'shipments.csv')
web_traffic = pd.read_csv(DATA/'web_traffic.csv')
sales       = pd.read_csv(DATA/'sales.csv', parse_dates=['Date'])
geography   = pd.read_csv(DATA/'geography.csv')
promotions  = pd.read_csv(DATA/'promotions.csv')

print('Loaded all tables.')


Loaded all tables.


## Q1 — Median inter-order gap (ngày)
**Cách làm:** sort theo (customer, date) → `groupby().diff()` → lấy median.
Đây là kỹ thuật chuẩn trong **RFM analysis** (Recency–Frequency–Monetary).

In [2]:
o = orders.sort_values(['customer_id', 'order_date'])
o['gap'] = o.groupby('customer_id')['order_date'].diff().dt.days
gaps = o['gap'].dropna()
print(f'Median: {gaps.median()} | Mean: {gaps.mean():.1f}')
# Đáp án: C) 180 ngày  (median 144 → gần 180 hơn 90)

Median: 144.0 | Mean: 285.6


## Q2 — Segment có gross margin trung bình cao nhất
Margin = (price − cogs) / price

In [3]:
products['margin'] = (products['price'] - products['cogs']) / products['price']
print(products.groupby('segment')['margin'].mean().sort_values(ascending=False))
# Đáp án: D) Standard

segment
Standard       0.313442
Premium        0.285377
All-weather    0.284176
Activewear     0.265600
Performance    0.263650
Balanced       0.258038
Trendy         0.240758
Everyday       0.236343
Name: margin, dtype: float64


## Q3 — Lý do trả hàng nhiều nhất cho danh mục Streetwear

In [4]:
sw_ids = products[products['category'] == 'Streetwear']['product_id']
r_sw = returns[returns['product_id'].isin(sw_ids)]
print(r_sw['return_reason'].value_counts())
# Đáp án: B) wrong_size

return_reason
wrong_size          7626
defective           4330
not_as_described    3854
changed_mind        3830
late_delivery       2159
Name: count, dtype: int64


## Q4 — Traffic source có bounce_rate trung bình thấp nhất

In [5]:
print(web_traffic.groupby('traffic_source')['bounce_rate'].mean().sort_values())
# Đáp án: C) email_campaign  (chênh lệch rất nhỏ giữa các source)

traffic_source
email_campaign    0.004458
social_media      0.004476
paid_search       0.004478
referral          0.004499
organic_search    0.004504
direct            0.004511
Name: bounce_rate, dtype: float64


## Q5 — % dòng order_items có promo_id không null

In [6]:
pct = order_items['promo_id'].notna().mean() * 100
print(f'% có promo: {pct:.2f}%')
# Đáp án: C) 39%

% có promo: 38.66%


## Q6 — Nhóm tuổi có số đơn TB / khách hàng cao nhất

In [7]:
c = customers.dropna(subset=['age_group'])
n_orders = orders.groupby('customer_id').size().rename('n_orders')
m = c.merge(n_orders, left_on='customer_id', right_index=True, how='left').fillna(0)
print(m.groupby('age_group')['n_orders'].mean().sort_values(ascending=False))
# Đáp án: A) 55+

age_group
55+      5.406851
45-54    5.357241
35-44    5.337343
25-34    5.245226
18-24    5.226656
Name: n_orders, dtype: float64


## Q7 — Vùng có doanh thu cao nhất (giai đoạn train)
**Lưu ý:** `sales.csv` không có cột region → phải tự tính revenue theo dòng:
`line_rev = unit_price * quantity - discount_amount`,
rồi join `order_items` × `orders` × `geography`.

In [8]:
oi = order_items.merge(orders[['order_id', 'zip', 'order_date']], on='order_id')
oi = oi.merge(geography[['zip', 'region']], on='zip')
oi['line_rev'] = oi['unit_price'] * oi['quantity'] - oi['discount_amount']
oi_train = oi[oi['order_date'] <= '2022-12-31']
print(oi_train.groupby('region')['line_rev'].sum().sort_values(ascending=False))
# Đáp án: C) East

region
East       7.291151e+09
Central    4.719491e+09
West       3.670227e+09
Name: line_rev, dtype: float64


## Q8 — Phương thức thanh toán phổ biến nhất với đơn cancelled

In [9]:
cancelled = orders[orders['order_status'] == 'cancelled']
print(cancelled['payment_method'].value_counts())
# Đáp án: A) credit_card

payment_method
credit_card      28452
cod              15468
paypal            7817
apple_pay         5190
bank_transfer     2535
Name: count, dtype: int64


## Q9 — Size có tỷ lệ trả hàng cao nhất

In [10]:
oi_p = order_items.merge(products[['product_id', 'size']], on='product_id')
r_p  = returns.merge(products[['product_id', 'size']], on='product_id')
rate = (r_p.groupby('size').size() / oi_p.groupby('size').size()).sort_values(ascending=False)
print(rate)
# Đáp án: A) S

size
S     0.056515
L     0.056250
M     0.055660
XL    0.055200
dtype: float64


## Q10 — Kế hoạch trả góp có giá trị TB cao nhất

In [11]:
print(payments.groupby('installments')['payment_value'].mean().sort_values(ascending=False))
# Đáp án: C) 6 kỳ
# Lưu ý: installments=2 có TB chỉ ~708 VND -> data quirk, đáng nhắc trong EDA

installments
6     24446.654403
3     24399.635486
12    24245.772694
1     24113.274166
2       708.473729
Name: payment_value, dtype: float64


## Tổng hợp đáp án
| Câu | Đáp án |
|-----|--------|
| Q1  | C — 180 ngày |
| Q2  | D — Standard |
| Q3  | B — wrong_size |
| Q4  | C — email_campaign |
| Q5  | C — 39% |
| Q6  | A — 55+ |
| Q7  | C — East |
| Q8  | A — credit_card |
| Q9  | A — S |
| Q10 | C — 6 kỳ |